In [1]:
import plotly.express as px
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import plotly.io as pio
import geopandas as gpd
from IPython.display import display, clear_output

In [2]:
f="../data/clean_bau_ws_30_50_80.csv"
df=pd.read_csv(f)
df.info()
pio.renderers.default = "notebook"

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   points_id      32 non-null     int64  
 1   location_name  32 non-null     str    
 2   latitude       32 non-null     float64
 3   longitude      32 non-null     float64
 4   bau30_ws_x_l   32 non-null     str    
 5   bau50_ws_x_l   32 non-null     str    
 6   bau80_ws_x_l   32 non-null     str    
 7   bau30_ws_x_r   32 non-null     float64
 8   bau50_ws_x_r   32 non-null     float64
 9   bau80_ws_x_r   32 non-null     float64
dtypes: float64(5), int64(1), str(4)
memory usage: 2.6 KB


In [3]:
# Reemplazar valores irreales por NaNs
df["bau30_ws_x_r"] = df["bau30_ws_x_r"].replace(9999, None)
# Convertir a numérico
df["bau30_ws_x_r"] = pd.to_numeric(df["bau30_ws_x_r"], errors="coerce")

# (opcional) limitar valores extremos
df["bau30_ws_x_r"] = df["bau30_ws_x_r"].clip(0, 5)

In [4]:
df

,points_id,location_name,latitude,longitude,bau30_ws_x_l,bau50_ws_x_l,bau80_ws_x_l,bau30_ws_x_r,bau50_ws_x_r,bau80_ws_x_r
0,1,Aguascalientes,21.8853,-102.2916,Extremely high (>80%),Extremely high (>80%),Extremely high (>80%),1.798984,2.090739,2.774931
1,2,Baja California,30.8406,-115.2838,Arid and low water use,Arid and low water use,Arid and low water use,1.000000,1.000000,1.000000
2,3,Baja California Sur,26.0444,-111.6661,Arid and low water use,Arid and low water use,Arid and low water use,1.000000,1.000000,1.000000
3,4,Campeche,19.8301,-90.5349,Low-medium (10-20%),Low-medium (10-20%),Medium-high (20-40%),0.101418,0.123944,0.288225
4,5,Chiapas,16.7569,-93.1292,Low (<10%),Low (<10%),Low-medium (10-20%),0.064338,0.070087,0.105702
5,6,Chihuahua,28.6330,-106.0691,Extremely high (>80%),Extremely high (>80%),Extremely high (>80%),1.586437,1.497713,2.687195
6,7,Ciudad de México,19.4326,-99.1332,Extremely high (>80%),Extremely high (>80%),Extremely high (>80%),5.000000,16.507452,13.853556
7,8,Coahuila,27.0587,-101.7068,Extremely high (>80%),Extremely high (>80%),Extremely high (>80%),1.813633,1.788625,2.104857
8,9,Colima,19.2452,-103.7241,Extremely high (>80%),Extremely high (>80%),Extremely high (>80%),1.463460,1.705520,2.117592
9,10,Durango,24.0277,-104.6532,Extremely high (>80%),Extremely high (>80%),Extremely high (>80%),1.566211,1.797931,2.343993


In [5]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# ── FUNCIONES DE CÁLCULO ───────────────────────────────────────

def compute_score(r):
    if r == 9999 or pd.isna(r): return 5
    if r <= 0: return 0
    return max(0, min(5, (np.log(r) - np.log(0.1)) / np.log(2) + 1))

def get_categoria_info(raw, score):
    if raw == 9999 or pd.isna(raw): return "Árido y bajo consumo", 95, ""
    elif score < 1:  return "<10%",   15, ""
    elif score < 2:  return "10-20%", 25, ""
    elif score < 3:  return "20-40%", 50, ""
    elif score < 4:  return "40-80%", 60, ""
    else:             return ">80%",  90, ""

def get_color_for_score(score):
    if score < 1:  return "#4fc97e"
    elif score < 2: return "#a8e6a3"
    elif score < 3: return "#f4d44d"
    elif score < 4: return "#f4a343"
    else:            return "#ef553b"

# ── FUNCIÓN COMPARTIDA: construye una figura de 3 gauges ─────────────────────
# Recibe `gauge_cfg`: lista de dicts con los parámetros de cada gauge.
# Así ambas pestañas comparten el mismo builder sin duplicar lógica.

def _build_triple_gauge(titulo, gauge_cfgs, progress, positions=[0.12, 0.5, 0.87]):
    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{'type': 'indicator'}] * 3],
        horizontal_spacing=0.10
    )
    for idx, cfg in enumerate(gauge_cfgs):
        fig.add_trace(go.Indicator(
            mode="gauge+number",
            value=round(cfg['value'] * progress, 2),
            number={'suffix': cfg['suffix'], 'font': {'size': 24, 'color': cfg['color']}},
            title={'text': f"<b>{cfg['label']}</b>", 'font': {'size': 16, 'color': cfg['color']}},
            domain={'row': 0, 'column': idx},
            gauge={
                'shape': "angular",
                'axis': cfg['axis'],
                'bar': {'color': cfg['color'], 'thickness': 0.15},
                'bgcolor': "white", 'borderwidth': 1, 'bordercolor': "#ddd",
                'steps': cfg['steps'],
                'threshold': {
                    'line': {'color': "darkred", 'width': 4},
                    'thickness': 0.8,
                    'value': cfg['value'] * progress
                }
            }
        ), row=1, col=idx + 1)

        # ✅ Anotación de categoría (antes faltaba fig.add_annotation)
        fig.add_annotation(
            y=-0.15, xref='paper', yref='paper',
            text=f"<b> Nota: Los % son solo aproximados para su mejor visualización, pero deben de ser interpretados como rangos representativos, no como valores puntuales</b>",
            showarrow=False, font={'size': 14}, align="center"
        )

    fig.update_layout(
        title={'text': titulo, 'x': 0.5, 'xanchor': 'center', 'font': {'size': 20}},
        height=400, font={'family': 'Arial'},
        margin={'t': 100, 'b': 80, 'l': 30, 'r': 30},
        paper_bgcolor="#fafafa", plot_bgcolor="white"
    )
    return fig

# ── PESTAÑA 1: Estrés hídrico (score 0–5) ────────────────────────────────────

COLORS = ['#2E86AB', '#A23B72', '#F18F01']
STEPS_SCORE = [
    {'range': [0, 1], 'color': "#4fc97e"}, {'range': [1, 2], 'color': "#a8e6a3"},
    {'range': [2, 3], 'color': "#f4d44d"}, {'range': [3, 4], 'color': "#f4a343"},
    {'range': [4, 5], 'color': "#ef553b"},
]
STEPS_PCT = [
    {'range': [0,  10], 'color': "#003f88"}, {'range': [10, 20], 'color': "#1a7fd4"},
    {'range': [20, 40], 'color': "#4da6ff"}, {'range': [40, 80], 'color':  "#99caff"},
    {'range': [80, 100], 'color': "#cce5ff"},
]

def _get_gauge_cfgs_score(row):
    cfgs = []
    for i, yr in enumerate(['2030', '2050', '2080']):
        raw   = row.get(f"bau{yr[2:]}_ws_x_r", 9999)
        score = compute_score(raw)
        cat, _, _ = get_categoria_info(raw, score)
        cfgs.append({
            'label': yr, 'color': COLORS[i],
            'value': score, 'suffix': '/5',
            'axis': {'range': [0, 5],
                      'tickvals': [0,1,2,3,4,5],
                      'ticktext': ['Muy\nBajo','Bajo','Bajo-\nMedio','Medio-\nAlto','Alto','Muy\nAlto'],
                      'tickfont': {'size': 8}, 'ticklen': 8, 'ticks': "outside"},
            'steps': STEPS_SCORE,
            'cat': cat, 'cat_color': get_color_for_score(score)
        })
    return cfgs

def _get_gauge_cfgs_pct(row):
    cfgs = []
    for i, yr in enumerate(['2030', '2050', '2080']):
        raw   = row.get(f"bau{yr[2:]}_ws_x_r", 9999)
        score = compute_score(raw)
        cat, nv, _ = get_categoria_info(raw, score)
        cfgs.append({
            'label': yr, 'color': COLORS[i],
            'value': nv, 'suffix': '%',
            'axis': {'range': [0, 100],
                      'tickvals': [0,10,20,40,80,100],
                      'ticktext': ['0%','10%','20%','40%','80%','100%'],
                      'tickfont': {'size': 8}, 'ticklen': 8, 'ticks': "outside"},
            'steps': STEPS_PCT,
            'cat': cat, 'cat_color': get_color_for_score(score)
        })
    return cfgs

def _animate(titulo, gauge_cfgs, animate):
    """Loop de animación compartido por ambas pestañas."""
    steps = 20 if animate else 1
    for step in range(steps + 1):
        clear_output(wait=True)
        _build_triple_gauge(titulo, gauge_cfgs, step / steps).show()
        if animate and step < steps:
            time.sleep(0.05)

def plot_gauges_animated(estado, animate=True):
    row = df[df["location_name"] == estado].iloc[0]
    titulo = (f"<b>📊 Estrés Hídrico: {estado}</b><br>"
              f"<sup>Evolución escenario BAU (Business As Usual)</sup>")
    _animate(titulo, _get_gauge_cfgs_score(row), animate)

def plot_detailed_gauges(estado, animate=True):
    row = df[df["location_name"] == estado].iloc[0]
    titulo = (f"<b>💧 % de agua extraída del total disponible: {estado}</b><br>"
              f"<sup>Evolución escenario BAU (Business As Usual)</sup>")
    _animate(titulo, _get_gauge_cfgs_pct(row), animate)

# ── WIDGET INTERACTIVO ────────────────────────────────────────────────────────

output   = widgets.Output()
dropdown = widgets.Dropdown(
    options=sorted(df["location_name"].unique()),
    description='🗺️ Estado:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)
animate_checkbox = widgets.Checkbox(
    value=True, description='🎬 Animación',
    style={'description_width': 'initial'}
)
view_toggle = widgets.ToggleButtons(
    options=['Estrés hídrico', '% de Agua extraída'],
    description='Visualización:',
    style={'description_width': 'initial'},
    button_style='info'
)

def update_plot(change=None):
    estado = dropdown.value
    animate = animate_checkbox.value
    with output:
        clear_output(wait=True)
        if view_toggle.value == 'Estrés hídrico':
            plot_gauges_animated(estado, animate=animate)
        else:
            plot_detailed_gauges(estado, animate=animate)  # ✅ sin fig.show() extra

dropdown.observe(lambda c: update_plot(), names='value')
animate_checkbox.observe(lambda c: update_plot(), names='value')
view_toggle.observe(lambda c: update_plot(), names='value')

controls = widgets.HBox([dropdown, animate_checkbox])
display(widgets.VBox([controls, view_toggle, output]))
update_plot()

## Visualización de estrés hídrico por estado según el escenario Business as Usual de los años 2030, 2050 y 2080.
Justificación: Creemos que esta gráfica es clave para el clímax de nuestro tablero, ya que nos permite visualizar de manera rápida e interactiva el estrés hídrico por estado de la República Mexicana. Ademàs, nos permite dimensionar la cantidad de agua que serà extraida en los siguientes años en funciòn del estrès hìdrico que sufre actualmente el estado ¿Observas cómo algunos estados se encuentran en crisis desde 2030? Esto nos ayuda a comprender que el enfoque tradicional de estrés hídrico ya no es suficiente para dimensionar la gravedad de la situación hídrica del país.

Es necesario adoptar un nuevo enfoque conceptual y un tipo de gestión más riguroso, el cual abordaremos en la siguiente visualización. En este, el agua se interpreta como “capital natural”, lo que permite analizar su consumo y disponibilidad desde una perspectiva similar a la financiera. Gracias a este enfoque, podemos categorizar a los estados de México en una situación de “bancarrota hídrica”.

In [6]:
# Cargar shapefile
gdf = gpd.read_file("../data/disponibilidad_de_agua_subterranea_09-11-2023.shp")

# Ver columnas
print(gdf.columns)
print(gdf.head())
gdf = gdf.rename(columns={
    "RECARGA_TO": "recarga",
    "VEAS": "extraccion",
    "DMA_POSITI": "disp_pos",
    "DMA_NEGATI": "disp_neg",
    "NOM_EDO": "estado",
    "NOM_ACUI": "acuifero"
})
gdf

Index(['CLV_EDO', 'NOM_EDO', 'CLV_REGION', 'NOM_REGION', 'CLV_ACUI',
       'NOM_ACUI', 'AREA_KM2', 'RECARGA_TO', 'DESCARGA_N', 'VCAS', 'VEALA',
       'VAPTYR', 'VAPRH', 'VEAS', 'DMA_POSITI', 'DMA_NEGATI', 'geometry'],
      dtype='str')
  CLV_EDO    NOM_EDO CLV_REGION                   NOM_REGION CLV_ACUI  \
0      32  ZACATECAS          7  CUENCAS CENTRALES DEL NORTE     3227   
1      32  ZACATECAS          7  CUENCAS CENTRALES DEL NORTE     3225   
2      32  ZACATECAS          7  CUENCAS CENTRALES DEL NORTE     3210   
3      32  ZACATECAS          7  CUENCAS CENTRALES DEL NORTE     3214   
4      32  ZACATECAS          7  CUENCAS CENTRALES DEL NORTE     3223   

                      NOM_ACUI  AREA_KM2  RECARGA_TO  DESCARGA_N        VCAS  \
0           GUADALUPE BAÑUELOS    290.33        12.1         0.0   12.296545   
1                       CALERA   2225.70        91.1         1.2  156.282584   
2                BENITO JUAREZ    350.50        18.1         0.0   21.486465   
3 

,CLV_EDO,estado,CLV_REGION,NOM_REGION,CLV_ACUI,acuifero,AREA_KM2,recarga,DESCARGA_N,VCAS,VEALA,VAPTYR,VAPRH,extraccion,disp_pos,disp_neg,geometry
0,32,ZACATECAS,7,CUENCAS CENTRALES DEL NORTE,3227,GUADALUPE BAÑUELOS,290.33,12.1,0.0,12.296545,0.000000,0.000000,0.198156,12.494701,0.000000,-0.394701,"POLYGON ((-102.44638 22.64909, -102.44107 22.6..."
1,32,ZACATECAS,7,CUENCAS CENTRALES DEL NORTE,3225,CALERA,2225.70,91.1,1.2,156.282584,0.000000,0.048004,0.830152,157.160740,0.000000,-67.260740,"POLYGON ((-102.64258 23.33527, -102.63898 23.2..."
2,32,ZACATECAS,7,CUENCAS CENTRALES DEL NORTE,3210,BENITO JUAREZ,350.50,18.1,0.0,21.486465,0.000000,0.000000,0.000000,21.486465,0.000000,-3.386465,"POLYGON ((-102.57755 22.69584, -102.58801 22.6..."
3,32,ZACATECAS,7,CUENCAS CENTRALES DEL NORTE,3214,AGUANAVAL,2804.42,84.5,0.0,173.037747,0.000000,0.000000,1.000995,174.038741,0.000000,-89.538741,"POLYGON ((-103.23971 22.72164, -103.22582 22.7..."
4,32,ZACATECAS,7,CUENCAS CENTRALES DEL NORTE,3223,GUADALUPE DE LAS CORRIENTES,4632.98,32.8,0.0,42.665038,0.000000,0.000000,0.137261,42.802299,0.000000,-10.002299,"POLYGON ((-102.24021 23.57484, -102.30331 23.5..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
648,5,COAHUILA,6,RIO BRAVO,0512,REGION CARBONIFERA,15753.71,84.1,39.1,36.461576,35.662534,0.005810,0.142378,72.272297,0.000000,-27.272297,"POLYGON ((-102.20495 28.90947, -102.18156 28.8..."
649,5,COAHUILA,6,RIO BRAVO,0503,CERRO COLORADO-LA PARTIDA,7201.24,9.6,0.0,0.714908,0.362099,0.000000,0.000731,1.077738,8.522262,0.000000,"POLYGON ((-102.26313 29.85908, -102.26343 29.8..."
650,5,COAHUILA,6,RIO BRAVO,0501,ALLENDE-PIEDRAS NEGRAS,12961.18,496.5,274.4,156.573961,80.362995,4.424811,0.182627,241.544394,0.000000,-19.444394,"POLYGON ((-100.58876 28.8605, -100.58771 28.85..."
651,5,COAHUILA,6,RIO BRAVO,0513,PALESTINA,3522.90,10.3,4.6,2.003137,1.280427,0.000000,0.101995,3.385559,2.314441,0.000000,"POLYGON ((-100.71715 28.94338, -100.91636 28.9..."


In [7]:
cols = ["recarga", "extraccion", "disp_pos", "disp_neg"]
gdf[cols] = gdf[cols].apply(lambda x: x.astype(float))
gdf.info()
df_estado = gdf.groupby("estado").agg({
    "recarga": "sum",
    "extraccion": "sum",
    "disp_pos": "sum", 
    "disp_neg": "sum"
}).reset_index()


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 653 entries, 0 to 652
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   CLV_EDO     653 non-null    str     
 1   estado      653 non-null    str     
 2   CLV_REGION  653 non-null    str     
 3   NOM_REGION  653 non-null    str     
 4   CLV_ACUI    653 non-null    str     
 5   acuifero    653 non-null    str     
 6   AREA_KM2    653 non-null    float64 
 7   recarga     653 non-null    float64 
 8   DESCARGA_N  653 non-null    float64 
 9   VCAS        653 non-null    float64 
 10  VEALA       653 non-null    float64 
 11  VAPTYR      653 non-null    float64 
 12  VAPRH       653 non-null    float64 
 13  extraccion  653 non-null    float64 
 14  disp_pos    653 non-null    float64 
 15  disp_neg    653 non-null    float64 
 16  geometry    653 non-null    geometry
dtypes: float64(10), geometry(1), str(6)
memory usage: 86.9 KB


In [8]:
estados   = df_estado["estado"]
extraccion = df_estado["extraccion"]
recarga    = df_estado["recarga"]

# ── Índices ─────────────────────────────────────────────────────────────────
# > 1 → recarga supera extracción (sostenible)
# < 1 → extracción supera recarga (déficit hídrico)
indice_sost = [r / e for r, e in zip(recarga, extraccion)]

# Déficit absoluto en Hm³ (0 para estados sostenibles)
deficit = [max(0, e - r) for e, r in zip(extraccion, recarga)]

# Tamaño de burbuja: escala logarítmica para que estados pequeños sean visibles
size_base = 8
bubble_size = [
    size_base + 28 * (np.log1p(d) / np.log1p(max(deficit) + 1))
    for d in deficit
]

# ── Color por zona de riesgo ─────────────────────────────────────────────────
# Rojo: bancarrota (índice < 0.5)  |  Amarillo: estrés (0.5–1.0)  |  Verde: sostenible
colores = [
    "#E24B4A" if i < 0.5
    else "#EF9F27" if i < 1.0
    else "#1D9E75"
    for i in indice_sost
]

# ── Hover: toda la información sin saturar el gráfico ───────────────────────
hover_text = [
    f"<b>{e}</b><br>"
    f"Extracción: {x:,.0f} Hm³<br>"
    f"Recarga:    {r:,.0f} Hm³<br>"
    f"Índice: {i:.2f} {'✅ sostenible' if i >= 1 else '⚠️ estrés' if i >= 0.5 else '🔴 bancarrota'}<br>"
    f"Déficit: {d:,.0f} Hm³"
    for e, x, r, i, d in zip(estados, extraccion, recarga, indice_sost, deficit)
]

# ── Figura ───────────────────────────────────────────────────────────────────
fig = go.Figure()

# Zona de bancarrota (debajo de la diagonal)
x_range = [5, 10000]
fig.add_trace(go.Scatter(
    x=x_range, y=x_range,
    fill="tozeroy",
    fillcolor="rgba(226, 75, 74, 0.08)",
    line=dict(color="rgba(226, 75, 74, 0.4)", dash="dash", width=1.5),
    name="Línea de equilibrio",
    hoverinfo="skip"
))

# Burbujas por estado
fig.add_trace(go.Scatter(
    x=extraccion,
    y=recarga,
    mode="markers",
    marker=dict(
        color=colores,
        size=bubble_size,
        opacity=0.82,
        line=dict(color="white", width=1)
    ),
    text=estados,
    hovertemplate="%{customdata}<extra></extra>",
    customdata=hover_text,
    name="Estados"
))

# ── Anotación zona de riesgo ─────────────────────────────────────────────────
fig.add_annotation(
    x=np.log10(500), y=np.log10(60),
    text="🔴 Zona de bancarrota hídrica",
    showarrow=False,
    font=dict(size=11, color="rgba(180,50,50,0.7)"),
    xref="x", yref="y"
)

fig.update_layout(
    title=dict(
        text=" Estados en Bancarrota Hídrica 2020",
        font=dict(size=15)
    ),
    xaxis=dict(
        title="Extracción (Hm³/año)",
        type="log",
        showgrid=True, gridcolor="rgba(0,0,0,0.06)"
    ),
    yaxis=dict(
        title="Recarga (Hm³/año)",
        type="log",
        showgrid=True, gridcolor="rgba(0,0,0,0.06)"
    ),
    plot_bgcolor="white",
    showlegend=False,
    hoverlabel=dict(font_size=12),
    height=560
)
# ── Contador de estados en bancarrota ────────────────────────────────────────
n_bancarrota = sum(1 for i in indice_sost if i < 1)
n_total      = len(estados)

fig.add_annotation(
    # Esquina inferior izquierda — coordenadas en fracción del área del plot
    xref="paper", yref="paper",
    x=0.02, y=0.04,
    xanchor="left", yanchor="bottom",
    text=(
        f"<b style='font-size:22px;color:#E24B4A'>{n_bancarrota}</b>"
        f"<span style='font-size:13px;color:#888'> / {n_total} estados<br>"
        f"en bancarrota hídrica</span>"
    ),
    showarrow=False,
    align="left",
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="rgba(226,75,74,0.35)",
    borderwidth=1,
    borderpad=8
)

fig.show()

# Bancarrota hídrica

Término acuñado por la ONU en enero de 2026 para describir una situación crítica en la que la humanidad ha agotado sus recursos de agua dulce a un ritmo superior al de su reposición natural. No se trata de una crisis temporal, sino de una pérdida potencialmente irreversible de los recursos hídricos, que afecta el equilibrio global y la capacidad de satisfacer necesidades básicas.

¿Cuál es el siguiente paso que nos gustaría abordar en la próxima etapa del hackatón? Nos preguntamos: ¿qué países están preparados para afrontar esta realidad de bancarrota hídrica? Buscaremos responder a esta pregunta utilizando datos abiertos del PDA de CONAGUA (Gobierno de México, marzo de 2026), donde se registra el nivel de vulnerabilidad social, económica y ambiental de cada estado de la República según su región hidrológica.

Asimismo, planeamos desarrollar, para la etapa de desenlace, cuatro ejes de acción que puede tomar un individuo para participar de manera individual, colectiva, institucional y gubernamental en la gestión del recurso hídrico de su región y que impacto podemos esperar de ello.

### Continuará…